# Random_Forest — otimização walk-forward

A seleção usa apenas os 80% de desenvolvimento. Em cada origem de validação o modelo é reajustado e prevê somente a próxima hora.

**Esta etapa é pesada e possui checkpoint em CSV.** Se interromper, rode novamente para continuar dos candidatos já concluídos.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image

here = Path.cwd().resolve()
group = next(
    (p for p in [here, *here.parents] if (p / "config.py").exists() and (p / "lib").is_dir()),
    None,
)
if group is None:
    raise RuntimeError("Não encontrei analyses/grupo2. Abra o notebook dentro do projeto.")
if str(group) not in sys.path:
    sys.path.insert(0, str(group))

import config
from lib import data as data_lib, features, eda_stl
from lib import models_v2 as models
from lib.nb_boot import require_file, save_json, load_json
data_lib.ensure_output_dirs()
print("outputs →", config.OUTPUT_DIR)


outputs → C:\Users\brunacardoso-ieg\OneDrive - Instituto Germinare\Área de Trabalho\Tech\Séries Temporais\trabalho_PLS_grupo2\pls-regration-comparation\analyses\grupo2\outputs


In [2]:
model_path = require_file(
    config.DATA_DIR / "modeling_frame.csv",
    "01_dados/03_feature_engineering.ipynb",
)
model_df = pd.read_csv(model_path, parse_dates=[config.DATETIME_COL])
protocol = models.protocol_summary(model_df)
print(
    f"linhas={len(model_df)} | validação={protocol['n_validation_origins']} origens "
    f"| teste={protocol['n_test_origins']} origens"
)


linhas=28636 | validação=258 origens | teste=521 origens


In [3]:
ranking_path = config.RESULT_DIR / "tuning_candidates_Random_Forest.csv"

winner, ranking = models.tune_random_forest(model_df, ranking_path)
save_json(winner, config.RESULT_DIR / "best_params_Random_Forest.json")
all_params_path = config.RESULT_DIR / "best_params.json"
all_params = load_json(all_params_path) if all_params_path.exists() else {}
all_params["Random_Forest"] = winner
save_json(all_params, all_params_path)
print("Vencedor:", winner)
display(ranking)



Random_Forest candidato 1/12: {'n_estimators': 600, 'max_depth': 24, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
Random_Forest: 20/258 origens | MAE parcial=113.893
Random_Forest: 40/258 origens | MAE parcial=125.144
Random_Forest: 60/258 origens | MAE parcial=128.247
Random_Forest: 80/258 origens | MAE parcial=149.920
Random_Forest: 100/258 origens | MAE parcial=140.388
Random_Forest: 120/258 origens | MAE parcial=170.891
Random_Forest: 140/258 origens | MAE parcial=178.293
Random_Forest: 160/258 origens | MAE parcial=172.997
Random_Forest: 180/258 origens | MAE parcial=181.003
Random_Forest: 200/258 origens | MAE parcial=186.669
Random_Forest: 220/258 origens | MAE parcial=189.948
Random_Forest: 240/258 origens | MAE parcial=198.530
Random_Forest: 258/258 origens | MAE parcial=198.895

Random_Forest candidato 2/12: {'n_estimators': 900, 'max_depth': 24, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
Random_Forest: 20/258 origens | M

,model,parameters,validation_mae,mean_seconds_per_origin,elapsed_seconds,convergence_rate,status,protocol_signature,depth_complexity,tree_complexity,mae_3
0,Random_Forest,"{""max_depth"": null, ""max_features"": ""sqrt"", ""m...",197.320995,3.426358,884.000439,1.0,ok,97047c16d77c1e22,inf,900,197.321
1,Random_Forest,"{""max_depth"": 20, ""max_features"": ""sqrt"", ""min...",198.098080,2.249346,580.331295,1.0,ok,97047c16d77c1e22,20.0,600,198.098
2,Random_Forest,"{""max_depth"": 24, ""max_features"": 0.3, ""min_sa...",198.870214,2.970962,766.508149,1.0,ok,97047c16d77c1e22,24.0,600,198.870
3,Random_Forest,"{""max_depth"": 24, ""max_features"": ""sqrt"", ""min...",198.894834,2.068875,533.769869,1.0,ok,97047c16d77c1e22,24.0,600,198.895
4,Random_Forest,"{""max_depth"": 24, ""max_features"": ""sqrt"", ""min...",199.120718,3.092651,797.904023,1.0,ok,97047c16d77c1e22,24.0,900,199.121
5,Random_Forest,"{""max_depth"": null, ""max_features"": ""sqrt"", ""m...",199.906603,2.004564,517.177567,1.0,ok,97047c16d77c1e22,inf,600,199.907
6,Random_Forest,"{""max_depth"": 24, ""max_features"": 0.7, ""min_sa...",199.969518,7.318079,1888.064511,1.0,ok,97047c16d77c1e22,24.0,500,199.970
7,Random_Forest,"{""max_depth"": 24, ""max_features"": ""sqrt"", ""min...",200.124179,1.993540,514.333222,1.0,ok,97047c16d77c1e22,24.0,600,200.124
8,Random_Forest,"{""max_depth"": 20, ""max_features"": 1.0, ""min_sa...",202.672884,4.634235,1195.632631,1.0,ok,97047c16d77c1e22,20.0,300,202.673
9,Random_Forest,"{""max_depth"": 20, ""max_features"": 1.0, ""min_sa...",203.025915,11.721223,3024.075436,1.0,ok,97047c16d77c1e22,20.0,500,203.026


O teste final não foi usado nesta escolha. Revise o ranking antes de executar o notebook de walk-forward final.